In [ ]:
!pip install -q transformers accelerate pillow opencv-python

In [ ]:
import getpass
api_key = getpass.getpass("please enter you gemini api key:  ")
import google.generativeai as genai
genai.configure(api_key=api_key)


In [1]:
!pip install -U "transformers==4.49.0" einops


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 62.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 56.7 MB/s eta 0:00:00:00:01
  Attempting uninstall: einops
    Found existing installation: einops 0.8.1
    Uninstalling einops-0.8.1:
      Successfully uninstalled einops-0.8.1
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transform

In [2]:
from transformers import AutoProcessor, AutoModelForCausalLM
import torch

model_id = "microsoft/Florence-2-large"

processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    trust_remote_code=True
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

print(f"loaded on {model.device}")


2026-03-06 08:16:57.778272: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772785018.025419      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772785018.091652      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772785018.628411      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772785018.628462      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772785018.628465      55 computation_placer.cc:177] computation placer alr

preprocessor_config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

processing_florence2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-large:
- processing_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/34.0 [00:00<?, ?B/s]

...

modeling_florence2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-large:
- modeling_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/1.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

loaded on cuda:0


In [3]:
import os
import json

page_id = "page_001"

base_dir = f"outputs/{page_id}"
lines_dir = f"{base_dir}/lines"

os.makedirs(lines_dir, exist_ok=True)

In [4]:
from PIL import Image

image_path = "/kaggle/input/datasets/shubhamshukla02/testdata/page1.png"

image = Image.open(image_path).convert("RGB")

image.save(f"{base_dir}/main_image.png")

In [5]:
import os

image_path = "/kaggle/input/datasets/shubhamshukla02/testdata/page1.png"

page_id = os.path.splitext(os.path.basename(image_path))[0]

In [6]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    trust_remote_code=True
).to("cuda").eval()


In [7]:
import torch
import cv2
import os
import json
import numpy as np
from PIL import Image

image_path = "/kaggle/input/datasets/shubhamshukla02/testdata/page1.png"
page_id = "page1"  

base_dir = f"outputs/{page_id}"
lines_dir = f"{base_dir}/lines"
os.makedirs(lines_dir, exist_ok=True)
image = Image.open(image_path).convert("RGB")
image.save(f"{base_dir}/main_image.png")
task_prompt = "<MORE_DETAILED_CAPTION>"

print(f" Florence-2...")
inputs = processor(text=task_prompt, images=image, return_tensors="pt").to("cuda", torch.float16)

with torch.no_grad():
    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=512,
        num_beams=3,
        do_sample=False
    )

generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
parsed_answer = processor.post_process_generation(
    generated_text, 
    task=task_prompt, 
    image_size=(image.width, image.height)
)

metadata_text = parsed_answer[task_prompt]
print(f"ok metadata: {metadata_text[:100]}...")
img_cv = cv2.imread(image_path)
img_gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
thresh = cv2.adaptiveThreshold(
    img_gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
    cv2.THRESH_BINARY_INV, 35, 15
)
projection = np.sum(thresh, axis=1)
threshold_value = (img_gray.shape[1] * 255) * 0.05 

lines = []
start = None

for i, val in enumerate(projection):
    if val > threshold_value and start is None:
        start = i
    elif val <= threshold_value and start is not None:
        if (i - start) > 10:  
            lines.append((start, i))
        start = None
line_paths = []
for idx, (y1, y2) in enumerate(lines):
    pad = 2
    y1_pad, y2_pad = max(0, y1-pad), min(img_gray.shape[0], y2+pad)
    
    crop = img_cv[y1_pad:y2_pad, :]
    line_filename = f"line_{idx:02d}.png"
    line_path = os.path.join(lines_dir, line_filename)
    cv2.imwrite(line_path, crop)
    line_paths.append(line_path)

metadata = {
    "page_id": page_id,
    "source_image": image_path,
    "description": metadata_text,
    "total_lines_found": len(line_paths),
    "lines": line_paths
}

with open(f"{base_dir}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"{len(line_paths)} lines. Data saved to {base_dir}/metadata.json")


 Florence-2...
ok metadata: The image is a photograph of an old, handwritten document. The document appears to be old and worn, ...
16 lines. Data saved to outputs/page1/metadata.json


In [8]:
import torch
import os
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from peft import PeftModel, PeftConfig
from safetensors.torch import load_file
model_id = "Qwen/Qwen2-VL-2B-Instruct"
adapter_path = "/kaggle/input/datasets/shubhamshukla02/qwen-finetuned/qwen_read_lora"
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("qwen done")
config = PeftConfig.from_pretrained(adapter_path)
model = PeftModel(model, config)
adapter_file = os.path.join(adapter_path, "adapter_model.safetensors")
state_dict = load_file(adapter_file)
new_state_dict = {}
for key, value in state_dict.items():
    new_key = key.replace("base_model.model.model", "base_model.model")
    new_state_dict[new_key] = value

model.load_state_dict(new_state_dict, strict=False)

try:
    processor = AutoProcessor.from_pretrained(adapter_path, trust_remote_code=True)
except Exception as e:
    print(f"issue: {e}")
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

model.eval()
print("-" * 30)
print("ok")
print("-" * 30)


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/429M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

------------------------------
ok
------------------------------


WE LOADED FINE TUNED QWEN

In [9]:
import json

metadata_path = "outputs/page1/metadata.json"

with open(metadata_path) as f:
    metadata = json.load(f)

line_paths = metadata["lines"]

print("lines", len(line_paths))

lines 16


In [10]:
from PIL import Image, ImageOps
import torch

results = []

for line_path in line_paths:
    image = Image.open(line_path).convert("RGB")
    w, h = image.size
    pad_w = max(0, 28 - w)
    pad_h = max(0, 28 - h)
    
    if pad_w > 0 or pad_h > 0:
        image = ImageOps.expand(image, (0, 0, pad_w, pad_h), fill="white")
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": "Transcribe the handwritten text."}
            ]
        }
    ]

    text_prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = processor(
        images=image,
        text=text_prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False 
        )
    prediction = processor.decode(
        output[0][len(inputs.input_ids[0]):], 
        skip_special_tokens=True
    )

    results.append({
        "line": line_path,
        "text": prediction
    })

    print(line_path, "→", prediction)


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.01` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.001` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


outputs/page1/lines/line_00.png → The handwritten text in the image is:

```
Información de afiliación hidalúrgica y minera de Sáxerre
unida a la Diputación de esta U. de Algeciras
alcalde del señor Alcalde de esta U. de Algeciras
y alcalde de la Diputación de esta U. de Algeciras
y alcalde de la Diputación de esta U. de Al
outputs/page1/lines/line_01.png → The handwritten text is:

"des de la configuración."
outputs/page1/lines/line_02.png → The handwritten text is:

"Undar de Muquiza nial y deino deita Villa."
outputs/page1/lines/line_03.png → Anuc Um pauco Comomas dia Suax; Hidio que
outputs/page1/lines/line_04.png → The handwritten text is:

"Soy hijo de Domingo de Muñoz y Álvarez."
outputs/page1/lines/line_05.png → The handwritten text is:

"de sauracowina na Difuntos házido iprocreado"
outputs/page1/lines/line_06.png → el matrimonio de los de Welles, Hijo por parte de la duena de Juan de Miguazza y la cabana de Sa
outputs/page1/lines/line_07.png → The handwritten text is:

"Haizi

Actual Result Predicted my our fine tuned Qwen

In [11]:
import json
import os

clean_results = []

for res in results:
    raw_text = res['text']
  
    if ":" in raw_text:
        actual_prediction = raw_text.split(":")[-1].strip()
    else:
        actual_prediction = raw_text.strip()
    
    actual_prediction = actual_prediction.replace('```', '').replace('"', '').strip()

    clean_results.append({
        "line": res['line'],
        "transcription": actual_prediction
    })

final_para = " ".join([item['transcription'] for item in clean_results if item['transcription']])

with open(f"{base_dir}/fine_tuned_output.json", "w") as f:
    json.dump(clean_results, f, indent=2)

with open(f"{base_dir}/actual_manuscript_text.txt", "w") as f:
    f.write(final_para)

print("qwen(fine tuned)output")
print(final_para)
print(f"\n fine tuned output{base_dir}/actual_manuscript_text.txt")


qwen(fine tuned)output
Información de afiliación hidalúrgica y minera de Sáxerre
unida a la Diputación de esta U. de Algeciras
alcalde del señor Alcalde de esta U. de Algeciras
y alcalde de la Diputación de esta U. de Algeciras
y alcalde de la Diputación de esta U. de Al des de la configuración. Undar de Muquiza nial y deino deita Villa. Anuc Um pauco Comomas dia Suax; Hidio que Soy hijo de Domingo de Muñoz y Álvarez. de sauracowina na Difuntos házido iprocreado el matrimonio de los de Welles, Hijo por parte de la duena de Juan de Miguazza y la cabana de Sa Haizicouli Loxma mufers aporla Mactina de Lower of Sauracouinita and Tonha of Canaregu manida y mula Lomos todos Verinos deuadha Villa quinta y Yo romos Hobbe hijos daleos dedan plaintext
que Christianos Cifos sin vara alguna de Moxor
Judios Moqués, Penitenciado por la Santa
Inquisición ni de otra Secta Nacuada, yantes Vien
dezendientes Léoninos alo Primero Poblado
us dierta M.H. Y.M.S. Provincia de Guipuz coa yonimarios y dependien

In [12]:
pip install -U google-genai

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 732.2/732.2 kB 15.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.5/236.5 kB 16.4 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.43.0
    Uninstalling google-auth-2.43.0:
      Successfully uninstalled google-auth-2.43.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 1.55.0
    Uninstalling google-genai-1.55.0:
      Successfully uninstalled google-genai-1.55.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires google-auth

In [13]:
import cv2

img = cv2.imread("/kaggle/input/datasets/shubhamshukla02/testdata/page1.png")
print(img.shape)

(942, 645, 3)


In [14]:
import cv2

img = cv2.imread("/kaggle/input/datasets/shubhamshukla02/testdata/page1.png")

scale = 2  
high_res = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)

cv2.imwrite("page_300dpi.jpg", high_res)

True

In [15]:
gray = cv2.cvtColor(high_res, cv2.COLOR_BGR2GRAY)
cv2.imwrite("page_gray.jpg", gray)

True

In [16]:
pip install scikit-image

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [17]:
from skimage.filters import threshold_sauvola
import numpy as np

window_size = 25
thresh = threshold_sauvola(gray, window_size=window_size)

binary = gray > thresh
binary = (binary * 255).astype("uint8")

cv2.imwrite("page_sauvola.jpg", binary)

True

In [18]:
pip install scikit-image

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [19]:
from skimage.morphology import skeletonize

skeleton = skeletonize(binary // 255)
skeleton = (skeleton * 255).astype("uint8")

cv2.imwrite("page_skeleton.jpg", skeleton)

True

In [20]:
import google.generativeai as genai
from PIL import Image

genai.configure(api_key=api_key)

model = genai.GenerativeModel("gemini-2.5-flash")

image = Image.open("page_sauvola.jpg")

prompt = """
You are a 16th-century Spanish paleography expert.

Transcribe the handwritten manuscript exactly.

Rules:
- Maintain original spelling
- Do not modernize words
- If strokes overlap, infer using grammatical context
- Preserve line breaks if possible
"""

response = model.generate_content([prompt, image])

print(response.text)

/usr/local/lib/python3.12/dist-packages/wrapt/importer.py:223: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  self.__wrapped__.exec_module(module)


Aquí está la transcripción del manuscrito, manteniendo la ortografía original y la estructura de las líneas:

ynformacion defiuacion Hidalguia p[ro]moüra de Baxen
con la annee.l Señor Alcalde de neüra V[ieja] y Algonax opocion
q[ue] Andres de Muguruza y Saca de sus Gracias y
los y Confirmacion.
Andres de Muguruza natural y Bezino de esta Villa
Anue vm parezco Comomas aia Lugar; Y digo q[ue]
Soy hijo Lexmo de Domingo de Muguruza y Anna
de Sagarrecoinia ya Difuntos Hazido y procraedo
en el Matrimonio Lexmo de ellos, Hizo por parte.
Laurena de Juan de Muguruza y Catalina de Ba-
larrzauri su Lexma muxer y por la Mariena de
Barr: de Sagarrecoinia y Josepha de Lanizagui
marido y muxer Lexmos todos Vecinos de esta
Villa quienes y lo somos Nobles hijos dalgo limpios
y Christianos Viejos sin Vara alguna de Moros
Judios ni Morizcos, Penitenciados por la Santa In-
quisicion ni de otra Secta Reprouada, y ante vm
Duzendú més Lexmos de los Primeros Poblado-
res de esta M.N.Y.M.L. Prouinzia de Guipuz

THE OUTPUT FROM THE GEMINI 2.5-FLASH (replace the api key with urs)

In [21]:
with open("gemini_raw.txt","w") as f:
    f.write(response.text)

In [22]:
with open("gemini_raw.txt","r") as f:
    raw_vlm_output = f.read()

In [23]:
import PIL.Image
from google import genai

client = genai.Client(api_key=api_key)

img = PIL.Image.open('/kaggle/input/datasets/shubhamshukla02/testdata/page1.png')

with open("gemini_raw.txt", "r", encoding="utf-8") as f:
    gemini_output = f.read()

qwen_path = "outputs/page1/actual_manuscript_text.txt"
with open(qwen_path, "r", encoding="utf-8") as f:
    qwen_output = f.read()

final_judge_prompt = f"""
You are an expert editor of early modern Spanish manuscripts.
Compare these two candidate transcriptions against the provided IMAGE.

Candidate 1 (Gemini): 
{gemini_output}

Candidate 2 (Qwen): 
{qwen_output}

Task: Use the image to resolve differences. 
Rules:
1. DO NOT modernize spelling.
2. Preserve historical orthography (ç, v for u, f for h).
3. Preserve line structure.
4. Output ONLY the final corrected text.
"""

response = client.models.generate_content(
    model="gemini-flash-latest",
    contents=[img, final_judge_prompt]
)

print(response.text)


Informacion defiliacion hidalguía y Limpieza de Sangre
unida ante el Señor Alcalde deesta Va d Elgoibar a pedimto
d Andres de Muguruza y Saca departidas de Baupto Casa-
dos y Confirmacion ==
        Andres de Muguruza nral y Vezino deesta Villa
        Ante Vm parezco Comomas aia Lugar; Y digo que
        Soy hijo Lexmo de Domingo de Muguruza y Antonia
        de Sagarrecomina ya Difuntos: Hazido y procreado
        enel Matrimonio Lexmo deellos, Nieto por parte
        Paterna de Juan de Muguruza y Catalina de Ba-
        llairteguí su Lexma muser y porla Materna de
        Barthme de Sagarrecomina y Josepha de Larrategui
        marido y muser Lexmos todos Vezinos deesta dha
        Villa quienes y Yo somos Nobles hijos dalgo de San-
        gre Christianos Viejos sin Raza alguna de Moros
        Judios: Hagonéss, Penitenciados porla Santa In-
        quisicion ni de otra Secta Reprouada, y antes Bien
        deszendiêntes Lexittimos delos Primeros Poblado-
        res deesta M.N. Y 

In [ ]:
THE FINAL LLM OUTPUT FROM THE (GEMINI OUTPUT+IMAGE+QWEN OUTPUT)

In [24]:

output_file_path = "final_answer"

with open(output_file_path, "w", encoding="utf-8") as f:
    f.write(response.text)

print(f"output saved: {output_file_path}")


output saved: final_answer


In [25]:
with open("/kaggle/working/final_answer.txt", "w", encoding="utf-8") as f:
    f.write(response.text)

import os
if os.path.exists("/kaggle/working/final_answer.txt"):
    print("saved")
else:
    print("prblm")


saved


In [28]:
import re
import Levenshtein

def get_clean_string(text):
    return re.sub(r'\s+', '', text)
try:
    with open("/kaggle/working/final_answer.txt", "r", encoding="latin-1") as f:
        pred_raw = f.read()

    with open("/kaggle/input/datasets/shubhamshukla02/ground-truth/ground_truth1.txt", "r", encoding="latin-1") as f:
        gt_raw = f.read()

    pred_clean = get_clean_string(pred_raw)
    gt_clean = get_clean_string(gt_raw)

    distance = Levenshtein.distance(gt_clean, pred_clean)
    cer = (distance / len(gt_clean)) * 100

    print(f"errors: {distance}")
    print(f"cer score: {cer:.2f}%")

except FileNotFoundError as e:
    print(f"error: {e}")


errors: 86
cer score: 8.92%


In [32]:
import re
from evaluate import load
bertscore = load("bertscore")
try:
    with open("/kaggle/working/final_answer.txt", "r", encoding="latin-1") as f:
        pred_raw = f.read()
    with open("/kaggle/input/datasets/shubhamshukla02/ground-truth/ground_truth1.txt", "r", encoding="latin-1") as f:
        gt_raw = f.read()
    predictions = [pred_raw]
    references = [gt_raw]
    results = bertscore.compute(predictions=predictions, references=references, 
                                lang="en", model_type="distilbert-base-uncased")

    print(f"F1 score: {results['f1'][0]:.4f}")
    print(f"Precision score: {results['precision'][0]:.4f}")
    print(f"Recall score: {results['recall'][0]:.4f}")

except FileNotFoundError as e:
    print(f"error is there: {e}")


F1 score: 0.9107
Precision score: 0.9080
Recall score: 0.9134


In [ ]:
import re
import Levenshtein

def get_clean_words(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text) 
    return text.split()

try:
    with open("/kaggle/working/final_answer.txt", "r", encoding="latin-1") as f:
        pred_raw = f.read()

    with open("/kaggle/input/datasets/shubhamshukla02/ground-truth/ground_truth1.txt", "r", encoding="latin-1") as f:
        gt_raw = f.read()
    pred_words = get_clean_words(pred_raw)
    gt_words = get_clean_words(gt_raw)
    distance = Levenshtein.distance(gt_words, pred_words)
    wer_score = (distance / len(gt_words)) * 100
    print(f"word error: {distance}")
    print(f"GT all words: {len(gt_words)}")
    print(f"wer score: {wer_score:.2f}%")

except FileNotFoundError as e:
    print(f"err: {e}")


In [30]:
!pip install evaluate bert-score


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.5 MB/s eta 0:00:00


In [27]:
!pip install python-Levenshtein

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 44.2 MB/s eta 0:00:0000:0100:01
